<a href="https://colab.research.google.com/github/Plumz17/PP_FinalProject2/blob/main/PP_FinalProject2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pattern Recognition Project - Banana Variety and Ripeness Classification using Traditional ML Feature Extraction Methods
Arranged by:
*   Anders Emmanuel Tan (24/541351/PA/22964)
*   Danar Fathurahman (24/538200/PA/22828)

Description: Pada projek ini, kami akan menyusun pipeline lengkap menggunakan teknik-teknik ekstraksi fitur tradisional untuk mengklasifikasi jenis dan tingkat kematangan pisang di Indonesia.

## 0. Image Acquisition (Loading the Dataset)
Description:

In [ ]:
# Import Important Libraries
import requests, os, zipfile, io, cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

In [ ]:
# Import Dataset
URL = "https://data.mendeley.com/public-api/zip/h6n5srjjyw/download/1"
os.makedirs("/content/banana", exist_ok=True)

r = requests.get(URL, stream=True, headers={"User-Agent": "Mozilla/5.0"})
total = int(r.headers.get("content-length", 0))

with open("/content/banana.zip", "wb") as f, tqdm(total=total, unit="B", unit_scale=True) as bar:
  for chunk in r.iter_content(chunk_size=1024*1024):
    f.write(chunk)
    bar.update(len(chunk))

print("\n")
print("Download Complete!")

100%|██████████| 715M/715M [00:28<00:00, 24.7MB/s]

Download Complete!


In [ ]:
# Extract outer zip
BASE_DIR = "/content/banana/BananaID An image dataset of banana varieties and/BananaID An image dataset of banana varieties and"
with zipfile.ZipFile("/content/banana.zip") as z:
    z.extractall(BASE_DIR)

# Extract all four inner zips
inner_zips = {
  "Banana Variety Dataset.zip":"variety_original",
  "Banana Ripeness Dataset.zip":"ripeness_original",
  "Augmented Banana Variety Dataset.zip":"variety_augmented",
  "Augmented Banana Ripeness Dataset.zip":"ripeness_augmented",
}

for zip_name, out_folder in inner_zips.items():
  zip_path = os.path.join(BASE_DIR, zip_name)
  out_path = os.path.join(BASE_DIR, out_folder)
  print(f"Extracting {zip_name}...")
  with zipfile.ZipFile(zip_path) as z:
    z.extractall(out_path)

print("Extraction complete!")

Extracting Banana Variety Dataset.zip...
Extracting Banana Ripeness Dataset.zip...
Extracting Augmented Banana Variety Dataset.zip...
Extracting Augmented Banana Ripeness Dataset.zip...
Extraction complete!


## 1. Preprocessing


In [ ]:
DATA_DIR = "/content/medicinal_plant/data/Medicinal_leaf_dataset"
IMG_SIZE = (384, 256)
class_names = sorted(os.listdir(DATA_DIR))
class_to_idx = {cls: idx for idx, cls in enumerate(class_names)}
images = []
labels = []
for class_name in tqdm(class_names, desc="Loading"):
  class_path = os.path.join(DATA_DIR, class_name)
  for img_file in os.listdir(class_path):
    img = cv2.imread(os.path.join(class_path, img_file))
    if img is None:
      continue
    #Resize image to the suggested size
    img = cv2.resize(img, IMG_SIZE)
    #Convert Color Model to RGB
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    images.append(img)
    labels.append(class_to_idx[class_name])

# Normalize each pixel to 0 - 1.0 and convert to numpy array
images = np.array(images, dtype=np.float32) / 255.0
labels = np.array(labels)

print(f"Images : {images.shape}")
print(f"Labels : {labels.shape}")
print(f"Classes: {class_names}")

Loading classes:   0%|          | 0/9 [00:00<?, ?it/s]


FileNotFoundError: [Errno 2] No such file or directory: '/content/banana/BananaID An image dataset of banana varieties and/BananaID An image dataset of banana varieties and/variety_original/ambon'

In [ ]:
def show(before, after, title2= "Sick"): # Helper function to check the before and after of the processing
  plt.figure(figsize=(10,5))

  # Show original
  plt.subplot(1,2,1)
  plt.imshow(before, cmap='gray', vmin=0, vmax=255) #vmin and max to preserve original image's contrast
  plt.title("Original")
  plt.axis("off")

  # Show processed
  plt.subplot(1,2,2)
  plt.imshow(after, cmap='gray', vmin=0, vmax=255)
  plt.title(title2)
  plt.axis("off")

  plt.show()

## 2. Image Enhancement